In [1]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_core.documents import Document
# from langchain_openai import OpenAIEmbeddings This is not free.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

C:\Users\Alpana\AppData\Local\Temp\ipykernel_25624\4115538631.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


True

In [2]:
chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [3]:
#Convert to Document objects for Langchain
documents = [Document(page_content=chunk,metadata={"source":f"chunk_{i}"}) for i,chunk in enumerate(chunks)]
print("Sample data :-")
for i,chunk in enumerate(chunks,1):
    print(f"{i}. {chunk}")

print("\n" + "="*80)

Sample data :-
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java 

In [21]:
print("Setting up vector retrieval")
embeddingModel = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    #model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore=Chroma.from_documents(documents=documents,
                                 embedding=embeddingModel,
                                 collection_metadata={"hnsw:space":"cosine"})


Setting up vector retrieval


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\grpc\fastapi\code\aishippinglabs\aihero\project\GithubAIAgent\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alpana\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
vectorRetriever = vectorstore.as_retriever(search_kwargs={"k":3})

#test semantic search
test_query = "space exploration company"

print(f"Testing: {test_query}")

test_docs=vectorRetriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing: space exploration company
Found: SpaceX develops Starship rockets for Mars missions.
Found: Google is a large technology company with global operations.
Found: NVIDIA designs Starship architecture for their new GPUs.


In [25]:
#BM25 retriever (Keyword search)
print("Setting up BM25 retriever")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k=2

test_query="Cybertruck"
print(f"Testing: {test_query}")
test_docs = bm25_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Setting up BM25 retriever
Testing: Cybertruck
Found: Cybertruck reservations exceeded company expectations.
Found: Tesla Cybertruck production ramp begins in 2024.


In [28]:
#Hybrid retriever - Combination of vector and BM25 retrievers
print("Setting up Hybrid retriever")
hybrid_retriever = EnsembleRetriever(
    retrievers=[vectorRetriever,bm25_retriever],
    weights=[0.7,0.3]
)

# test_query = "purchase cost 7.5 billion"
test_query = "electric vehicle manufacturing Cybertruck"
retrieved_chunks = hybrid_retriever.invoke(test_query)
for i,doc in enumerate(retrieved_chunks,1):
    print(f"{i}: {doc.page_content}")

Setting up Hybrid retriever
1: Cybertruck reservations exceeded company expectations.
2: Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3: Tesla Cybertruck production ramp begins in 2024.


In [36]:
#Combine the query and the relevant document  contents
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}
Please provide a clear, helpful answer using only the information from these documents. 
If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash",
    temperature=0
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input)
]

result = llm.invoke(messages)

print(f"Result: {result.content}")

d:\grpc\fastapi\code\aishippinglabs\aihero\project\GithubAIAgent\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Result: [{'type': 'text', 'text': 'Based on the provided documents:\n\n* **Electric Vehicle Leadership & Facilities:** Tesla continues to lead in electric vehicles and has announced new manufacturing facilities.\n* **Cybertruck Production:** The production ramp for the Tesla Cybertruck begins in 2024.\n* **Demand:** Cybertruck reservations have exceeded company expectations.', 'extras': {'signature': 'EpMOCpAOARFNMg+SSOf36oHr+S6dq5wVwpKhdMRvOacjo45Wqo4EkRPcsswSWF06FPgyFo98HXR90wd70JInw9dDwjDjFHozAeqMNCZ8XlwotOQMPiGdklxyimNf88o+FZ0IVJCgJNb7gxqKwD8a2TABL+jh8SUnMS1AUWny9Qz+4FUr89FjEi1JQO6kxNMLVoxWbRmwwQoOi+sh0iqJ07XNB9e4SVGKcvjGRNmkmTCOsTzFKf1aJkFkoH9t7jwZCdy0cBIx6ykkVdf/ckzYksxZX8tv22Bl87M8XPkMz/1rak0tCfg7X2Had079UkDDxcZDB8DomtCSvIiOhLKWA3w/6vhmrxRamGhBt0dOma+yFqv1TQhI6xcvzSDkDX2FMyqFoWd7NgthBKWYtjsg1D1fbz3AnRxsQ68CgFjO9VSncNhrjoMzApqwD2qj6ui42Yb+tJ773fBGNUxp5XFcQopZiSIERbWw2xbq8kBJcZzyOe1/TPTZQURt8I19A8bF5XvBwBkv84Crkzw3Mm8bn0W3jG3Sv5lrC13TtQ8KnVuLhU44GWixB9sdHFdasWjZcbmenmrkwVKBYXSJdwm